In [ ]:
import re
REPO_URL = "https://github.com/Dung34/vector-kv-cache.git"
COMMIT = ""  # dán SHA 40 ký tự
SCRIPT = "experiments/phase5_header_demo.py"
NONCE = ""  # người dùng tự đặt
TAG = "phase5_kaggle"
TRANSFORMERS_VERSION = "5.20.1"
assert re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT), "Hãy điền COMMIT là SHA-1 gồm 40 ký tự hex."
assert NONCE, "Hãy điền NONCE trước khi chạy notebook."
assert re.fullmatch(r"[A-Za-z0-9_.-]+", TAG), "Hãy điền TAG chỉ gồm chữ, số, _, ., -."

In [ ]:
import os, subprocess
subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
subprocess.run(["git", "checkout", COMMIT], cwd="repo", check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd="repo", check=True, capture_output=True, text=True).stdout.strip()
assert head == COMMIT
print(f"HEAD={head}")
print(subprocess.run(["git", "status", "--porcelain"], cwd="repo", check=True, capture_output=True, text=True).stdout)
os.chdir("repo")

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"transformers=={TRANSFORMERS_VERSION}", "accelerate"], check=True)
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
result = subprocess.run(["bash", "experiments/run_exp.sh", SCRIPT, NONCE, TAG, "python"])
if result.returncode != 0:
    raise RuntimeError(f"runner failed with exit code {result.returncode}")

In [ ]:
from pathlib import Path
import hashlib, zipfile
roots = [Path(name) for name in ("logs", "results") if Path(name).exists()]
files = [path for root in roots for path in sorted(root.rglob("*")) if path.is_file()]
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
manifest = Path("MANIFEST.txt")
manifest.write_text(f"COMMIT={COMMIT}\nNONCE={NONCE}\n" + "\n".join(f"{p}\t{p.stat().st_size}\t{sha256(p)}" for p in files), encoding="utf-8")
output = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
archive_path = output / f"evidence_{TAG}.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files: archive.write(path, path.as_posix())
    archive.write(manifest, "MANIFEST.txt")
print(f"evidence={archive_path} sha256={sha256(archive_path)}")